# Notebook 08 — Minimum Cost to EPC C

**Research question:** For each London social rented property currently below EPC C, what would it cost to reach the 2030 target — and which boroughs face the highest burden?

## Methodology & limitation

The EPC recommendations file (`recommendations-*.csv`) records each assessor improvement recommendation with:
- `improvement_item` — priority order (1 = highest priority)
- `improvement_id` — improvement type code
- `improvement_summary_text` — description
- `indicative_cost` — national cost band e.g. `'£500 - £1,500'`

**What the data does NOT include:** a per-improvement EPC score uplift. The full RdSAP calculation model knows how many points each improvement contributes, but this is not published in the open data. The `potential_energy_efficiency` column on the certificate gives the score achievable if *all* recommendations are implemented — but not the incremental contribution of each one.

**Approach used here:**
1. Use `potential_energy_efficiency` to determine whether a property *can* reach C at all
2. For properties that can reach C: sum costs of *all* recommendations as an **upper bound** on cost to reach C (some improvements may not be strictly necessary — the true minimum cost is ≤ this figure)
3. Flag properties where the upper bound exceeds the government's £10,000 spend exemption cap
4. Aggregate to borough level for three cost scenarios (optimistic / central / pessimistic)

This is a conservative overestimate. A more precise model would require per-improvement EPC uplift data (available in full SAP software, not in the public dataset).

In [ ]:
import glob
import re
import numpy as np
import pandas as pd

pd.set_option('display.float_format', '{:,.0f}'.format)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 40)

BRONZE_EPC  = '../data/bronze/epc_raw'
GOLD        = '../data/gold'

EPC_C_THRESHOLD   = 69   # minimum score for band C
EXEMPTION_CAP     = 10_000  # £10k government spend exemption cap

print('Libraries loaded')

## 1. Load London social rented certificates

Re-reading from bronze to get `potential_energy_efficiency`, which was not included in the silver ingestion (silver was built for current-condition analysis only).

In [ ]:
cert_files = sorted(glob.glob(f'{BRONZE_EPC}/certificates-*.csv'))
print(f'Certificate files: {len(cert_files)}')

CERT_COLS = [
    'certificate_number', 'local_authority_label', 'tenure', 'region',
    'current_energy_efficiency', 'potential_energy_efficiency',
    'current_energy_rating', 'construction_age_band', 'property_type'
]

chunks = []
for f in cert_files:
    df = pd.read_csv(f, usecols=CERT_COLS, dtype=str, low_memory=False)
    # Filter to London social rented
    df = df[
        (df['region'] == 'E12000007') &
        (df['tenure'].str.upper().isin(['RENTAL (SOCIAL)', 'SOCIAL RENTED', 'RENTED (SOCIAL)']))
    ]
    chunks.append(df)

certs = pd.concat(chunks, ignore_index=True)
certs = certs.rename(columns={'local_authority_label': 'borough'})
certs['current_energy_efficiency']   = pd.to_numeric(certs['current_energy_efficiency'],  errors='coerce')
certs['potential_energy_efficiency'] = pd.to_numeric(certs['potential_energy_efficiency'], errors='coerce')

print(f'London social rented certs: {len(certs):,}')
print(f'Boroughs: {certs["borough"].nunique()}')
print(f'Null current score: {certs["current_energy_efficiency"].isna().sum():,}')
print(f'Null potential score: {certs["potential_energy_efficiency"].isna().sum():,}')

## 2. Load recommendations and join to certificates

In [ ]:
rec_files = sorted(glob.glob(f'{BRONZE_EPC}/recommendations-*.csv'))
print(f'Recommendation files: {len(rec_files)}')

REC_COLS = ['certificate_number', 'improvement_item', 'improvement_id',
            'improvement_summary_text', 'indicative_cost']

recs = pd.concat(
    [pd.read_csv(f, usecols=REC_COLS, dtype=str, low_memory=False) for f in rec_files],
    ignore_index=True
)
recs['improvement_item'] = pd.to_numeric(recs['improvement_item'], errors='coerce')
recs['improvement_id']   = pd.to_numeric(recs['improvement_id'],   errors='coerce')

print(f'Total recommendation rows: {len(recs):,}')

# Join to London social rented certs only
joined = recs.merge(
    certs[['certificate_number', 'borough', 'current_energy_efficiency',
           'potential_energy_efficiency', 'current_energy_rating',
           'construction_age_band', 'property_type']],
    on='certificate_number', how='inner'
)

print(f'Recommendations for London social rented: {len(joined):,}')
print(f'Distinct properties with recommendations: {joined["certificate_number"].nunique():,}')

## 3. Parse indicative costs (3 scenarios)

In [ ]:
def parse_cost(s, which='mid'):
    """Parse '£800 - £1,200' into low / mid / high numeric values."""
    if pd.isna(s):
        return None
    nums = [float(n.replace(',', '')) for n in re.findall(r'[\d,]+', str(s))]
    if not nums:
        return None
    if which == 'low':
        return nums[0]
    if which == 'high':
        return nums[-1]   # same as low if only one number
    # mid
    return sum(nums) / len(nums)

joined['cost_low']  = joined['indicative_cost'].apply(parse_cost, which='low')
joined['cost_mid']  = joined['indicative_cost'].apply(parse_cost, which='mid')
joined['cost_high'] = joined['indicative_cost'].apply(parse_cost, which='high')

print('Cost parsing check (sample of distinct cost strings):')
joined[['indicative_cost', 'cost_low', 'cost_mid', 'cost_high']] \
    .drop_duplicates('indicative_cost') \
    .sort_values('cost_mid') \
    .head(15)

## 4. Property-level gap analysis

For each property currently below EPC C (score < 69), we calculate:
- `gap` — points needed to reach C
- `can_reach_c` — whether potential score >= 69
- `total_cost_*` — sum of all recommendations (upper bound on cost to reach C)
- `exceeds_cap` — whether upper bound cost > £10,000 exemption threshold

In [ ]:
# Work only with below-C properties
below_c_certs = certs[
    certs['current_energy_efficiency'].notna() &
    (certs['current_energy_efficiency'] < EPC_C_THRESHOLD)
].copy()

below_c_certs['gap'] = EPC_C_THRESHOLD - below_c_certs['current_energy_efficiency']
below_c_certs['can_reach_c'] = (
    below_c_certs['potential_energy_efficiency'].notna() &
    (below_c_certs['potential_energy_efficiency'] >= EPC_C_THRESHOLD)
)

print(f'Properties below EPC C: {len(below_c_certs):,}')
print(f'  Can reach C (potential >= 69): {below_c_certs["can_reach_c"].sum():,} ({below_c_certs["can_reach_c"].mean()*100:.1f}%)')
print(f'  Cannot reach C even with all improvements: {(~below_c_certs["can_reach_c"]).sum():,} ({(~below_c_certs["can_reach_c"]).mean()*100:.1f}%)')
print(f'  Unknown (null potential score): {below_c_certs["potential_energy_efficiency"].isna().sum():,}')
print()
print('Gap distribution (points needed to reach C):')
print(below_c_certs['gap'].describe().round(1))

In [ ]:
# Sum all recommendation costs per below-C property
below_c_recs = joined[joined['current_energy_efficiency'] < EPC_C_THRESHOLD].copy()

prop_costs = (
    below_c_recs
    .groupby('certificate_number')
    .agg(
        n_recommendations=('improvement_item', 'count'),
        total_cost_low=('cost_low',  'sum'),
        total_cost_mid=('cost_mid',  'sum'),
        total_cost_high=('cost_high', 'sum'),
    )
    .reset_index()
)

# Merge back with cert-level info
prop_summary = below_c_certs.merge(prop_costs, on='certificate_number', how='left')
prop_summary['has_recs'] = prop_summary['n_recommendations'].notna()
prop_summary['exceeds_cap_optimistic']  = prop_summary['total_cost_low']  > EXEMPTION_CAP
prop_summary['exceeds_cap_central']     = prop_summary['total_cost_mid']  > EXEMPTION_CAP
prop_summary['exceeds_cap_pessimistic'] = prop_summary['total_cost_high'] > EXEMPTION_CAP

print(f'Below-C properties with at least one recommendation: {prop_summary["has_recs"].sum():,} ({prop_summary["has_recs"].mean()*100:.1f}%)')
print(f'Below-C properties with no recommendations on record: {(~prop_summary["has_recs"]).sum():,}')
print()

can_reach = prop_summary[prop_summary['can_reach_c'] & prop_summary['has_recs']]
print(f'Can-reach-C properties with recommendations: {len(can_reach):,}')
print(f'  Exceeds £10k cap (central estimate): {can_reach["exceeds_cap_central"].sum():,} ({can_reach["exceeds_cap_central"].mean()*100:.1f}%)')
print(f'  Within £10k cap (central estimate):  {(~can_reach["exceeds_cap_central"]).sum():,} ({(~can_reach["exceeds_cap_central"]).mean()*100:.1f}%)')

## 5. Borough-level summary

In [ ]:
borough_summary = (
    prop_summary
    .groupby('borough')
    .agg(
        total_below_c=('certificate_number', 'count'),
        can_reach_c=('can_reach_c', 'sum'),
        pct_can_reach_c=('can_reach_c', 'mean'),
        avg_gap=('gap', 'mean'),
    )
    .reset_index()
)
borough_summary['cannot_reach_c'] = borough_summary['total_below_c'] - borough_summary['can_reach_c']
borough_summary['pct_cannot_reach_c'] = 1 - borough_summary['pct_can_reach_c']
borough_summary['pct_can_reach_c'] = (borough_summary['pct_can_reach_c'] * 100).round(1)
borough_summary['pct_cannot_reach_c'] = (borough_summary['pct_cannot_reach_c'] * 100).round(1)
borough_summary['avg_gap'] = borough_summary['avg_gap'].round(1)

print('=== Borough summary: below-C properties and reachability ===')
print('Sorted by % that CANNOT reach C — these are the hardest cases.')
display(
    borough_summary
    .sort_values('pct_cannot_reach_c', ascending=False)
    .reset_index(drop=True)
)

## 6. Cost to reach C — borough aggregations (3 scenarios)

**Important caveat:** These figures sum the cost of *all* recommendations for each below-C property. Since we cannot determine which subset of improvements is strictly necessary to cross 69 (per-improvement EPC uplift is not in the public data), this is an **upper bound**. The true minimum cost to reach C is lower — potentially significantly so for properties that need only 1–2 measures.

For properties that **cannot** reach C even with all improvements, the cost is still shown — this represents the total spend before a £10k exemption would be claimed.

In [ ]:
borough_costs = (
    prop_summary[prop_summary['has_recs']]
    .groupby('borough')
    .agg(
        properties_with_recs=('certificate_number', 'count'),
        # cost per home
        avg_cost_per_home_low=('total_cost_low',  'mean'),
        avg_cost_per_home_mid=('total_cost_mid',  'mean'),
        avg_cost_per_home_high=('total_cost_high', 'mean'),
        # total borough cost
        total_cost_low_m=('total_cost_low',  'sum'),
        total_cost_mid_m=('total_cost_mid',  'sum'),
        total_cost_high_m=('total_cost_high', 'sum'),
        # exemption
        n_exceeds_cap_central=('exceeds_cap_central', 'sum'),
        pct_exceeds_cap_central=('exceeds_cap_central', 'mean'),
    )
    .reset_index()
)

# Convert totals to £m
for col in ['total_cost_low_m', 'total_cost_mid_m', 'total_cost_high_m']:
    borough_costs[col] = (borough_costs[col] / 1_000_000).round(1)

borough_costs['pct_exceeds_cap_central'] = (borough_costs['pct_exceeds_cap_central'] * 100).round(1)
for col in ['avg_cost_per_home_low', 'avg_cost_per_home_mid', 'avg_cost_per_home_high']:
    borough_costs[col] = borough_costs[col].round(0)

print('=== Cost to reach EPC C — borough scenarios (upper bound) ===')
print('Sorted by avg cost per home (central) — shows retrofit difficulty.')
print('Note: London labour costs typically run 15-30% above national average.')
print('These indicative costs are national-rate estimates and will understate true London costs.')
print()
display(
    borough_costs[[
        'borough', 'properties_with_recs',
        'avg_cost_per_home_low', 'avg_cost_per_home_mid', 'avg_cost_per_home_high',
        'total_cost_low_m', 'total_cost_mid_m', 'total_cost_high_m',
        'n_exceeds_cap_central', 'pct_exceeds_cap_central'
    ]]
    .sort_values('avg_cost_per_home_mid', ascending=False)
    .rename(columns={
        'avg_cost_per_home_low':  'per_home_low_£',
        'avg_cost_per_home_mid':  'per_home_mid_£',
        'avg_cost_per_home_high': 'per_home_high_£',
        'total_cost_low_m':  'total_low_£m',
        'total_cost_mid_m':  'total_mid_£m',
        'total_cost_high_m': 'total_high_£m',
        'n_exceeds_cap_central':   'n_over_10k',
        'pct_exceeds_cap_central': 'pct_over_10k',
    })
    .reset_index(drop=True)
)

## 7. Exemption analysis — properties that cannot reach C within £10k

In [ ]:
# Three categories per borough:
# 1. Can reach C, within £10k  — straightforward retrofit
# 2. Can reach C, over £10k    — can reach C but expensive; may still be done with grant support
# 3. Cannot reach C at all     — structural limitation; exemption required

def classify(row):
    if not row['can_reach_c']:
        return 'Cannot reach C (exemption needed)'
    if row['exceeds_cap_central']:
        return 'Can reach C — over £10k'
    return 'Can reach C — within £10k'

prop_summary['category'] = prop_summary.apply(classify, axis=1)

exemption_summary = (
    prop_summary
    .groupby(['borough', 'category'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Add % columns
total_col = exemption_summary.iloc[:, 1:].sum(axis=1)
for cat in ['Can reach C — within £10k', 'Can reach C — over £10k', 'Cannot reach C (exemption needed)']:
    if cat in exemption_summary.columns:
        exemption_summary[f'pct_{cat}'] = (exemption_summary[cat] / total_col * 100).round(1)

print('=== Exemption analysis by borough ===')
print('Sorted by % cannot reach C — these boroughs have the most structurally hard cases.')
display(
    exemption_summary
    .sort_values('pct_Cannot reach C (exemption needed)', ascending=False)
    .reset_index(drop=True)
)

## 8. Quick wins — properties just below C (score 65–68)

In [ ]:
# Properties at 65-68: need only a small uplift — likely crossable with 1-2 cheap measures
quick_wins = prop_summary[
    prop_summary['current_energy_efficiency'].between(65, 68) &
    prop_summary['can_reach_c']
].copy()

print(f'Properties at 65-68 ("quick wins" — just below C): {len(quick_wins):,}')
print(f'  Within £10k cap (central): {(~quick_wins["exceeds_cap_central"]).sum():,} ({(~quick_wins["exceeds_cap_central"]).mean()*100:.1f}%)')
print()

quick_wins_by_borough = (
    quick_wins
    .groupby('borough')
    .agg(
        n_quick_wins=('certificate_number', 'count'),
        avg_cost_mid=('total_cost_mid', 'mean'),
    )
    .reset_index()
    .sort_values('n_quick_wins', ascending=False)
)
quick_wins_by_borough['avg_cost_mid'] = quick_wins_by_borough['avg_cost_mid'].round(0)

print('=== Quick-win properties (score 65-68) by borough ===')
print('These are the easiest C-band wins — prioritise these first in any retrofit programme.')
display(quick_wins_by_borough.head(20).reset_index(drop=True))

## 9. Construction age band breakdown

Pre-1919 solid-brick stock has the highest per-property cost and the lowest rate of being able to reach C. This cell quantifies that directly.

In [ ]:
age_summary = (
    prop_summary[prop_summary['has_recs']]
    .groupby('construction_age_band')
    .agg(
        n_properties=('certificate_number', 'count'),
        pct_can_reach_c=('can_reach_c', 'mean'),
        avg_gap=('gap', 'mean'),
        avg_cost_per_home_mid=('total_cost_mid', 'mean'),
        pct_over_10k=('exceeds_cap_central', 'mean'),
    )
    .reset_index()
)
age_summary['pct_can_reach_c']     = (age_summary['pct_can_reach_c'] * 100).round(1)
age_summary['pct_over_10k']        = (age_summary['pct_over_10k'] * 100).round(1)
age_summary['avg_gap']             = age_summary['avg_gap'].round(1)
age_summary['avg_cost_per_home_mid'] = age_summary['avg_cost_per_home_mid'].round(0)

print('=== Cost to reach C by construction age band ===')
print('Validates that older stock is more expensive to retrofit — pre-1919 is the hardest.')
display(age_summary.sort_values('avg_cost_per_home_mid', ascending=False).reset_index(drop=True))

## 10. Most common first improvement for below-C properties

What do assessors most commonly recommend as priority 1 for properties that need to reach C? This shows what the bottleneck measure is by borough.

In [ ]:
from functools import reduce

# Priority 1 recommendations for below-C properties only
priority_1 = joined[
    (joined['current_energy_efficiency'] < EPC_C_THRESHOLD) &
    (joined['improvement_item'] == 1)
].copy()

print('=== Most common priority-1 improvement for below-C London social stock ===')
display(
    priority_1
    .groupby('improvement_summary_text')
    .agg(
        times_recommended=('certificate_number', 'count'),
        avg_cost_mid=('cost_mid', 'mean'),
    )
    .sort_values('times_recommended', ascending=False)
    .assign(avg_cost_mid=lambda x: x['avg_cost_mid'].round(0))
    .head(15)
    .reset_index()
)

# Top priority-1 improvement per priority borough
priority_boroughs = ['Barking and Dagenham', 'Haringey', 'Lambeth', 'Hammersmith and Fulham', 'Enfield']

print('\n=== Most common priority-1 improvement — top 5 priority boroughs ===')
top1_per_borough = (
    priority_1[priority_1['borough'].isin(priority_boroughs)]
    .groupby(['borough', 'improvement_summary_text'])
    .size()
    .reset_index(name='count')
    .sort_values(['borough', 'count'], ascending=[True, False])
    .groupby('borough')
    .head(3)
    .reset_index(drop=True)
)
display(top1_per_borough)

## 11a. Prorated cost estimate — better than 'sum all recs'

The upper-bound approach (cell 10) sums every recommendation cost, even for properties that only need a small uplift. A property needing 3 of a possible 12 points should not be charged for 100% of the improvement package.

**Prorated estimate:** `total_rec_costs × (points_needed / total_possible_uplift)`

This assumes costs are distributed proportionally across the uplift — still an approximation, but significantly more accurate than the naive upper bound for properties close to the C threshold.

In [ ]:
prop_summary['uplift_available'] = (
    prop_summary['potential_energy_efficiency'] - prop_summary['current_energy_efficiency']
)

# Prorated fraction: what share of the uplift package do we need?
prop_summary['pct_uplift_needed'] = np.where(
    prop_summary['can_reach_c'] & (prop_summary['uplift_available'] > 0),
    (prop_summary['gap'] / prop_summary['uplift_available']).clip(0, 1),
    np.nan
)

prop_summary['prorated_cost_low']  = prop_summary['total_cost_low']  * prop_summary['pct_uplift_needed']
prop_summary['prorated_cost_mid']  = prop_summary['total_cost_mid']  * prop_summary['pct_uplift_needed']
prop_summary['prorated_cost_high'] = prop_summary['total_cost_high'] * prop_summary['pct_uplift_needed']

can_reach = prop_summary[prop_summary['can_reach_c'] & prop_summary['has_recs']]
ub_med = can_reach['total_cost_mid'].median()
pr_med = can_reach['prorated_cost_mid'].median()

print(f'Can-reach-C properties with recommendations: {len(can_reach):,}')
print(f'\nUpper bound (all recs):')
print(f'  Median cost to C (central): £{ub_med:,.0f}')
print(f'\nProrated estimate (proportional to uplift fraction needed):')
print(f'  Median cost to C (central): £{pr_med:,.0f}')
print(f'  Reduction vs upper bound:   {(1 - pr_med/ub_med)*100:.0f}%')
print(f'\nNote: prorated is still conservative — it assumes uniform cost distribution across')
print(f'uplift points. The ML model below attempts a more realistic attribution.')

## 12. ML uplift model — per-recommendation EPC point attribution

**The problem:** The public EPC data gives us the *total* uplift from the full recommendation package (`potential - current`), but not the marginal contribution of each individual measure.

**ML approach:**
1. Build a certificate-level feature matrix: property attributes + which recommendation types are present (multi-hot)
2. Target: `potential_energy_efficiency − current_energy_efficiency`
3. Train a Random Forest regressor
4. Use SHAP (TreeExplainer) to attribute predicted uplift to each recommendation feature
5. Use SHAP-derived per-recommendation point estimates to find the **cheapest path to C** via greedy selection

**Important caveat:** SHAP values reflect the model's learned association between having a recommendation and seeing higher uplift — not a causal per-recommendation score. Properties with more recommendations are already at lower efficiency, creating confounds. Treat these as *indicative estimates*, not ground truth.

The ONS Data Science Campus found ML models achieve ~67% of properties within ±5 SAP points using EPC-derived features — good enough for portfolio-level analysis, not precise enough for individual property guarantees.

In [ ]:
import re
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# ── 1. Top improvement types as multi-hot features ────────────────────────────
top_n = 20
top_imps = (
    joined['improvement_summary_text']
    .value_counts()
    .head(top_n)
    .index.tolist()
)

def safe_col(s):
    return 'rec_' + re.sub(r'[^a-z0-9]+', '_', s.lower().strip())[:38]

col_map = {imp: safe_col(imp) for imp in top_imps}
print('Top improvement types used as ML features:')
for imp, col in col_map.items():
    print(f'  {col}: {imp}')


In [ ]:
# ── 2. Build certificate-level feature matrix ─────────────────────────────────
# Multi-hot: does this certificate have this recommendation type?
cert_multihot = (
    joined[joined['improvement_summary_text'].isin(top_imps)]
    .assign(col=lambda df: df['improvement_summary_text'].map(col_map), val=1)
    .pivot_table(index='certificate_number', columns='col', values='val', aggfunc='max', fill_value=0)
    .reset_index()
)

# Certificate-level property features
cert_feats = (
    certs[
        certs['current_energy_efficiency'].notna() &
        certs['potential_energy_efficiency'].notna()
    ]
    [['certificate_number', 'current_energy_efficiency', 'potential_energy_efficiency',
      'property_type', 'construction_age_band']]
    .copy()
)
cert_feats['uplift'] = (
    cert_feats['potential_energy_efficiency'] - cert_feats['current_energy_efficiency']
)
cert_feats = cert_feats[cert_feats['uplift'] >= 0]  # discard anomalous records

for cat_col in ['property_type', 'construction_age_band']:
    le = LabelEncoder()
    cert_feats[cat_col + '_enc'] = le.fit_transform(cert_feats[cat_col].fillna('Unknown'))

ml_data = cert_feats.merge(cert_multihot, on='certificate_number', how='left').fillna(0)

rec_feature_cols = list(col_map.values())
property_cols    = ['current_energy_efficiency', 'property_type_enc', 'construction_age_band_enc']
all_features     = property_cols + rec_feature_cols

X = ml_data[all_features]
y = ml_data['uplift']

print(f'ML dataset: {len(ml_data):,} certificates, {len(all_features)} features')
print(f'Target (EPC uplift) — mean: {y.mean():.1f}, median: {y.median():.1f}, std: {y.std():.1f}')


In [ ]:
# ── 3. Train Random Forest regressor ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=150, max_depth=12, min_samples_leaf=20,
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
pct5 = (np.abs(y_pred - y_test) <= 5).mean() * 100

print(f'Model performance on held-out 20%:')
print(f'  MAE:                   {mae:.2f} EPC points')
print(f'  R²:                    {r2:.3f}')
print(f'  Within ±5 EPC points:  {pct5:.1f}% of properties')
print(f'\nBaseline (predict mean): MAE = {mean_absolute_error(y_test, np.full_like(y_test, y_train.mean())):.2f}')


In [ ]:
# ── 4. SHAP attribution — estimated points per recommendation type ─────────────
try:
    import shap
    sample_size = min(10_000, len(ml_data))
    X_shap = X.sample(sample_size, random_state=42)

    explainer   = shap.TreeExplainer(rf)
    shap_vals   = explainer.shap_values(X_shap)

    # Mean absolute SHAP for recommendation features only
    rec_indices = [list(all_features).index(c) for c in rec_feature_cols]
    shap_rec    = np.abs(shap_vals[:, rec_indices]).mean(axis=0)

    rec_shap_df = pd.DataFrame({
        'improvement_type':       [c.replace('rec_', '').replace('_', ' ').title() for c in rec_feature_cols],
        'improvement_summary_text': top_imps,
        'est_points_contribution': shap_rec.round(2),
    }).sort_values('est_points_contribution', ascending=False).reset_index(drop=True)

    # Join to recommendation costs for cost-efficiency table
    avg_costs = (
        joined[joined['improvement_summary_text'].isin(top_imps)]
        .groupby('improvement_summary_text')[['cost_low', 'cost_mid', 'cost_high']]
        .mean()
        .reset_index()
    )
    rec_cost_eff = rec_shap_df.merge(avg_costs, on='improvement_summary_text', how='left')
    rec_cost_eff['£_per_est_point'] = (rec_cost_eff['cost_mid'] / rec_cost_eff['est_points_contribution']).round(0)

    print('=== Per-recommendation estimated EPC point contribution and cost efficiency ===')
    print('Lower £/point = more cost-effective path to EPC C')
    print()
    display(
        rec_cost_eff[['improvement_type', 'est_points_contribution', 'cost_low', 'cost_mid', 'cost_high', '£_per_est_point']]
        .rename(columns={'cost_low':'avg_cost_low_£','cost_mid':'avg_cost_mid_£','cost_high':'avg_cost_high_£'})
    )
    shap_available = True

except ImportError:
    print('shap not installed — run: pip install shap>=0.44')
    print('Feature importance (sklearn fallback):')
    fi = pd.DataFrame({
        'feature': all_features,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False).head(20)
    display(fi)
    rec_cost_eff = None
    shap_available = False


In [ ]:
# ── 5. Greedy cheapest-path optimizer ─────────────────────────────────────────
# For each below-C property with recs and can_reach_c:
#   rank its recommendations by cost_mid / estimated_points (cheapest per point)
#   greedily select until cumulative points >= gap
#   sum costs of selected recs → ML-optimised minimum cost estimate

if shap_available and rec_cost_eff is not None:
    # Build lookup: improvement_type → estimated_points
    points_lookup = rec_cost_eff.set_index('improvement_summary_text')['est_points_contribution'].to_dict()

    below_c_with_recs = joined[
        (joined['current_energy_efficiency'] < EPC_C_THRESHOLD) &
        (joined['potential_energy_efficiency'] >= EPC_C_THRESHOLD)
    ].copy()
    below_c_with_recs['est_points'] = below_c_with_recs['improvement_summary_text'].map(points_lookup).fillna(0.5)
    below_c_with_recs['cost_per_point'] = np.where(
        below_c_with_recs['est_points'] > 0,
        below_c_with_recs['cost_mid'] / below_c_with_recs['est_points'],
        np.inf
    )
    below_c_with_recs['gap'] = EPC_C_THRESHOLD - below_c_with_recs['current_energy_efficiency']

    # Sort by cost efficiency within each property
    below_c_with_recs = below_c_with_recs.sort_values(
        ['certificate_number', 'cost_per_point']
    )

    # Greedy selection: cumulative points until gap is covered
    below_c_with_recs['cum_points'] = below_c_with_recs.groupby('certificate_number')['est_points'].cumsum()
    below_c_with_recs['gap_covered'] = below_c_with_recs['cum_points'] >= below_c_with_recs['gap']

    # First rec where gap is covered — all recs up to and including that one are "needed"
    first_covered = (
        below_c_with_recs[below_c_with_recs['gap_covered']]
        .groupby('certificate_number')['cum_points']
        .idxmin()
    )
    needed_mask = below_c_with_recs.index.isin(
        below_c_with_recs.loc[
            below_c_with_recs.groupby('certificate_number').cumcount() <=
            below_c_with_recs.groupby('certificate_number').cumcount().transform(
                lambda x: x[below_c_with_recs.loc[x.index, 'gap_covered'].idxmax() if x.index.isin(first_covered.values).any() else len(x)]
                if len(x) > 0 else 0
            )
        ].index
    )

    # Vectorised greedy: mark recs up to and including the first gap-covering rec
    below_c_with_recs['row_rank'] = below_c_with_recs.groupby('certificate_number').cumcount()
    coverage_rank = (
        below_c_with_recs[below_c_with_recs['gap_covered']]
        .groupby('certificate_number')['row_rank']
        .min()
        .rename('coverage_rank')
    )
    below_c_with_recs = below_c_with_recs.join(coverage_rank, on='certificate_number')
    below_c_with_recs['selected'] = below_c_with_recs['row_rank'] <= below_c_with_recs['coverage_rank']

    ml_costs = (
        below_c_with_recs[below_c_with_recs['selected']]
        .groupby('certificate_number')
        .agg(
            ml_cost_low =('cost_low',  'sum'),
            ml_cost_mid =('cost_mid',  'sum'),
            ml_cost_high=('cost_high', 'sum'),
            n_recs_needed=('improvement_summary_text', 'count'),
        )
        .reset_index()
    )

    prop_summary = prop_summary.merge(ml_costs, on='certificate_number', how='left')
    prop_summary['ml_exceeds_cap'] = prop_summary['ml_cost_mid'] > EXEMPTION_CAP

    can_reach = prop_summary[prop_summary['can_reach_c'] & prop_summary['has_recs'] & prop_summary['ml_cost_mid'].notna()]
    print(f'Properties with ML-optimised path estimate: {len(can_reach):,}')
    print(f'\nCost-to-C comparison (median, central estimate):')
    print(f'  Upper bound (all recs):           £{can_reach["total_cost_mid"].median():,.0f}')
    print(f'  Prorated estimate:                £{can_reach["prorated_cost_mid"].median():,.0f}')
    print(f'  ML greedy cheapest path:          £{can_reach["ml_cost_mid"].median():,.0f}')
    print(f'\nML estimate: {can_reach["ml_cost_mid"].median() / can_reach["total_cost_mid"].median() * 100:.0f}% of upper bound')
    print(f'Avg recs needed to reach C: {can_reach["n_recs_needed"].mean():.1f} of {can_reach["n_recommendations"].mean():.1f} total')
else:
    print('Skipping greedy optimiser — SHAP not available.')


## 13. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
BOROUGH_FONT = 7

# ── Chart 1: Reachability by borough (stacked bar) ───────────────────────────
already_c = certs[certs['current_energy_efficiency'] >= EPC_C_THRESHOLD].groupby('borough').size().rename('already_c')
reach_summary = (
    prop_summary
    .groupby('borough')
    .apply(lambda df: pd.Series({
        'can_reach': df['can_reach_c'].sum(),
        'cannot_reach': (~df['can_reach_c']).sum(),
    }))
    .join(already_c, how='outer')
    .fillna(0)
    .astype(int)
)
reach_summary['total'] = reach_summary.sum(axis=1)
reach_pct = reach_summary[['already_c', 'can_reach', 'cannot_reach']].div(reach_summary['total'], axis=0) * 100
reach_pct = reach_pct.sort_values('cannot_reach', ascending=True)

fig, ax = plt.subplots(figsize=(12, 9))
bottom = np.zeros(len(reach_pct))
colours = ['#2ca02c', '#1f77b4', '#d62728']
labels  = ['Already ≥ EPC C', 'Below C — can reach C with listed improvements', 'Below C — cannot reach C (exemption likely needed)']
for col, colour, label in zip(['already_c', 'can_reach', 'cannot_reach'], colours, labels):
    ax.barh(reach_pct.index, reach_pct[col], left=bottom, color=colour, label=label, height=0.7)
    bottom += reach_pct[col].values

ax.set_xlabel('% of social rented stock', fontsize=10)
ax.set_title('EPC C reachability by borough — London social rented stock\n(sorted by % that cannot reach C)', fontsize=11, fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.tick_params(axis='y', labelsize=BOROUGH_FONT)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.savefig('outputs/08_reachability_by_borough.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 08_reachability_by_borough.png')


In [ ]:
# ── Chart 2: Cost-to-C comparison: upper bound vs prorated vs ML (top boroughs) ──
cost_comp_cols = {
    'Upper bound\n(all recs)':  'total_cost_mid',
    'Prorated\nestimate':      'prorated_cost_mid',
}
if 'ml_cost_mid' in prop_summary.columns:
    cost_comp_cols['ML greedy\ncheapest path'] = 'ml_cost_mid'

borough_cost_comp = (
    prop_summary[prop_summary['can_reach_c'] & prop_summary['has_recs']]
    .groupby('borough')[list(cost_comp_cols.values())]
    .median()
    .reset_index()
    .sort_values('total_cost_mid', ascending=False)
    .head(20)
)

x = np.arange(len(borough_cost_comp))
width = 0.8 / len(cost_comp_cols)
fig, ax = plt.subplots(figsize=(14, 6))
cmap = ['#d62728', '#ff7f0e', '#1f77b4']
for i, (label, col) in enumerate(cost_comp_cols.items()):
    ax.bar(x + i * width, borough_cost_comp[col], width=width, label=label, color=cmap[i])

ax.set_xticks(x + width * (len(cost_comp_cols) - 1) / 2)
ax.set_xticklabels(borough_cost_comp['borough'], rotation=45, ha='right', fontsize=BOROUGH_FONT)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'£{v:,.0f}'))
ax.set_ylabel('Median cost to EPC C per property (central estimate)', fontsize=9)
ax.set_title('Cost-to-C estimates: upper bound vs prorated vs ML greedy optimiser\n(top 20 boroughs by cost, below-C properties that can reach C)', fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('outputs/08_cost_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 08_cost_comparison.png')


In [ ]:
# ── Chart 3: Age band — cost and reachability ─────────────────────────────────
age_order = [
    'England and Wales: before 1900',
    'England and Wales: 1900-1929',
    'England and Wales: 1930-1949',
    'England and Wales: 1950-1966',
    'England and Wales: 1967-1975',
    'England and Wales: 1976-1982',
    'England and Wales: 1983-1990',
    'England and Wales: 1991-1995',
    'England and Wales: 1996-2002',
    'England and Wales: 2003-2006',
    'England and Wales: 2007-2011',
    'England and Wales: 2012 onwards',
]
age_plot = (
    age_summary
    [age_summary['construction_age_band'].isin(age_order)]
    .set_index('construction_age_band')
    .reindex(age_order)
    .dropna(subset=['avg_cost_per_home_mid'])
)
short_labels = [
    'Pre-1900', '1900–29', '1930–49', '1950–66', '1967–75', '1976–82',
    '1983–90', '1991–95', '1996–02', '2003–06', '2007–11', '2012+'
][:len(age_plot)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
bars = ax1.bar(range(len(age_plot)), age_plot['avg_cost_per_home_mid'], color='#1f77b4')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'£{v:,.0f}'))
ax1.set_ylabel('Avg cost to C (central, £)')
ax1.set_title('Cost and reachability by construction age band\n(London social rented stock below EPC C)', fontweight='bold')

ax2.bar(range(len(age_plot)), age_plot['pct_can_reach_c'], color='#2ca02c')
ax2.axhline(50, color='red', linestyle='--', linewidth=0.8, label='50% threshold')
ax2.set_ylabel('% can reach EPC C')
ax2.set_ylim(0, 105)
ax2.set_xticks(range(len(age_plot)))
ax2.set_xticklabels(short_labels[:len(age_plot)], rotation=30, ha='right', fontsize=8)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('outputs/08_age_band_cost.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 08_age_band_cost.png')


## 11. Save gold outputs

In [ ]:
import os
os.makedirs(f'{GOLD}/minimum_cost_to_c', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Borough cost scenarios
borough_costs.to_parquet(f'{GOLD}/minimum_cost_to_c/borough_cost_scenarios.parquet', index=False)
borough_costs.to_csv('outputs/minimum_cost_to_c_borough.csv', index=False)
print('Saved borough_cost_scenarios')

# Borough reachability + exemption summary
borough_summary_full = borough_summary.merge(borough_costs, on='borough', how='left')
borough_summary_full.to_parquet(f'{GOLD}/minimum_cost_to_c/borough_reachability.parquet', index=False)
borough_summary_full.to_csv('outputs/minimum_cost_to_c_reachability.csv', index=False)
print('Saved borough_reachability')

# Quick wins
quick_wins_by_borough.to_csv('outputs/minimum_cost_to_c_quick_wins.csv', index=False)
print('Saved quick_wins_by_borough')

# Age band
age_summary.to_csv('outputs/minimum_cost_to_c_by_age_band.csv', index=False)
print('Saved age_band_breakdown')

print('\nAll outputs saved.')

# ML-optimised cost estimates
if 'ml_cost_mid' in prop_summary.columns:
    ml_summary = (
        prop_summary[prop_summary['can_reach_c'] & prop_summary['ml_cost_mid'].notna()]
        .groupby('borough')
        .agg(
            n_properties=('certificate_number', 'count'),
            median_ml_cost_low =('ml_cost_low',  'median'),
            median_ml_cost_mid =('ml_cost_mid',  'median'),
            median_ml_cost_high=('ml_cost_high', 'median'),
            n_over_10k=('ml_exceeds_cap', 'sum'),
            pct_over_10k=('ml_exceeds_cap', 'mean'),
        )
        .reset_index()
    )
    ml_summary['pct_over_10k'] = (ml_summary['pct_over_10k'] * 100).round(1)
    for col in ['median_ml_cost_low', 'median_ml_cost_mid', 'median_ml_cost_high']:
        ml_summary[col] = ml_summary[col].round(0)
    ml_summary.to_parquet(f'{GOLD}/minimum_cost_to_c/borough_ml_cost_scenarios.parquet', index=False)
    ml_summary.to_csv('outputs/minimum_cost_to_c_ml.csv', index=False)
    print('Saved borough_ml_cost_scenarios')

if rec_cost_eff is not None:
    rec_cost_eff.to_csv('outputs/recommendation_cost_efficiency.csv', index=False)
    print('Saved recommendation_cost_efficiency.csv')


## Summary of findings

Key outputs from this notebook:

- **Can/cannot reach C:** What % of below-C stock in each borough can reach EPC C at all with the recommended improvements
- **Cost scenarios:** Per-home and total borough cost (optimistic / central / pessimistic) — upper bound since all recommendations are summed
- **Exemption exposure:** % of properties where the upper-bound cost exceeds the £10,000 government exemption cap
- **Quick wins:** Properties at score 65–68 that need only a small uplift — highest priority for a retrofit programme
- **Age band breakdown:** Confirms pre-1919 stock has the highest per-home cost and the lowest rate of being able to reach C
- **Bottleneck measure:** The most common priority-1 improvement for below-C stock by borough

**Next steps / extensions:**
- Obtain per-improvement EPC score uplifts from full SAP software to compute true minimum cost (not upper bound)
- Apply London labour cost uplift (15–30%) to all indicative costs
- Cross-reference with borough_wall_type (notebook 06 gold) — boroughs with high solid-wall % will have higher exemption rates
- Model phased compliance: which properties hit the 2030 deadline vs 2039 second metric deadline